# Modeling Inflation Dynamics in Argentina

This notebook reproduces the econometric analysis presented in the repository README. The objective is to examine the statistical relationship between monthly exchange-rate movements and changes in monthly inflation in Argentina using an iterative time-series modeling strategy. The analysis characterizes statistical associations and temporal dynamics, not structural causality.

## 1. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch, breaks_cusumolsresid, recursive_olsresiduals
from arch import arch_model

## 2. Data Loading and Preparation

In [ ]:
datos = pd.read_csv("../data/inflacion_tipo_cambio_niveles.csv")
datos["Periodo"] = pd.to_datetime(datos["Periodo"])
datos = datos.sort_values("Periodo").reset_index(drop=True)
print(datos.shape)
display(datos.head())
print(datos.columns)

In [ ]:
datos["delta_inflacion"] = datos["inflacion_mensual"].diff()
display(datos[["Periodo", "inflacion_mensual", "variacion_dolar", "delta_inflacion"]].head(10))

## 3. Time-Series Visualization

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(datos["Periodo"], datos["inflacion_mensual"], linewidth=1.8)
plt.xlabel("Period")
plt.ylabel("Monthly inflation (%)")
plt.title("Monthly Inflation Rate in Argentina")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(datos["Periodo"], datos["delta_inflacion"], linewidth=1.5)
plt.axhline(0, linewidth=1, alpha=0.6)
plt.xlabel("Period")
plt.ylabel("Change in monthly inflation (p.p.)")
plt.title("Change in Monthly Inflation Rate")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Stationarity Tests

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"\n{name}")
    print(f"ADF statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print(f"Lags used: {result[2]}")
    print(f"Observations: {result[3]}")
    print("Critical values:")
    for level, value in result[4].items():
        print(f"  {level}: {value:.4f}")

def kpss_test(series, name):
    result = kpss(series.dropna(), regression="c", nlags="auto")
    print(f"\n{name}")
    print(f"KPSS statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print(f"Lags used: {result[2]}")
    print("Critical values:")
    for level, value in result[3].items():
        print(f"  {level}: {value:.4f}")

In [ ]:
adf_test(datos["inflacion_mensual"], "Monthly inflation")
kpss_test(datos["inflacion_mensual"], "Monthly inflation")
adf_test(datos["delta_inflacion"], "Change in monthly inflation")
kpss_test(datos["delta_inflacion"], "Change in monthly inflation")
adf_test(datos["variacion_dolar"], "Monthly exchange-rate variation")

## 5. ACF and PACF of the Stationary Inflation Series

In [ ]:
serie = datos["delta_inflacion"].dropna()
plot_acf(serie, lags=24)
plt.title("ACF - Change in Monthly Inflation")
plt.show()
plot_pacf(serie, lags=24, method="ywm")
plt.title("PACF - Change in Monthly Inflation")
plt.show()

## 6. Initial ARIMA Model Selection

In [ ]:
serie = datos["inflacion_mensual"].dropna()
resultados_arima = []
for p in range(3):
    for q in range(3):
        try:
            m = ARIMA(serie, order=(p,1,q)).fit()
            resultados_arima.append({"p":p,"d":1,"q":q,"AIC":m.aic,"BIC":m.bic})
        except Exception as exc:
            print(f"ARIMA({p},1,{q}) failed: {exc}")
comparacion_arima = pd.DataFrame(resultados_arima).sort_values("AIC")
display(comparacion_arima)

In [ ]:
modelo_212 = ARIMA(serie, order=(2,1,2)).fit()
modelo_210 = ARIMA(serie, order=(2,1,0)).fit()
print(modelo_212.summary())
print(modelo_210.summary())

AIC initially favors ARIMA(2,1,2), while BIC favors ARIMA(2,1,0). ARIMA(2,1,2) is retained as an exploratory specification before incorporating the exchange-rate variable.

## 7. Initial ARIMAX Specification

In [ ]:
df_arimax = datos[["inflacion_mensual", "variacion_dolar"]].dropna()
y_arimax = df_arimax["inflacion_mensual"]
X_arimax = df_arimax[["variacion_dolar"]]
modelo_arimax = ARIMA(y_arimax, exog=X_arimax, order=(2,1,2)).fit()
print(modelo_arimax.summary())

## 8. ARCH-LM Test on Initial ARIMAX Residuals

In [ ]:
residuos_arimax = modelo_arimax.resid.dropna()
rows=[]
for lag in [3,6,12]:
    r=het_arch(residuos_arimax, nlags=lag)
    rows.append({"Lag":lag,"LM Statistic":r[0],"p-value":r[1]})
display(pd.DataFrame(rows))

## 9. ARCH/GARCH Model Comparison

In [ ]:
df_vol = datos[["inflacion_mensual", "variacion_dolar"]].copy()
df_vol["delta_inflacion"] = df_vol["inflacion_mensual"].diff()
df_vol = df_vol.dropna()
y = df_vol["delta_inflacion"]
X = df_vol[["variacion_dolar"]]
modelo_arch1 = arch_model(y, x=X, mean="ARX", lags=0, vol="ARCH", p=1, dist="normal").fit(disp="off")
modelo_arch2 = arch_model(y, x=X, mean="ARX", lags=0, vol="ARCH", p=2, dist="normal").fit(disp="off")
modelo_garch11 = arch_model(y, x=X, mean="ARX", lags=0, vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
comparacion_vol = pd.DataFrame({
    "Model":["ARCH(1)","ARCH(2)","GARCH(1,1)"],
    "AIC":[modelo_arch1.aic,modelo_arch2.aic,modelo_garch11.aic],
    "BIC":[modelo_arch1.bic,modelo_arch2.bic,modelo_garch11.bic],
    "LogLik":[modelo_arch1.loglikelihood,modelo_arch2.loglikelihood,modelo_garch11.loglikelihood]
}).sort_values("AIC")
display(comparacion_vol)

## 10. Volatility Model Diagnostics

In [ ]:
for name, model in [("ARCH(1)",modelo_arch1),("ARCH(2)",modelo_arch2),("GARCH(1,1)",modelo_garch11)]:
    resid_std=model.std_resid.dropna()
    print(f"\n=== {name} ===")
    print("Ljung-Box: standardized residuals")
    display(acorr_ljungbox(resid_std,lags=[6,12],return_df=True))
    print("Ljung-Box: squared standardized residuals")
    display(acorr_ljungbox(resid_std**2,lags=[6,12],return_df=True))
    arch_rows=[]
    for lag in [3,6,12]:
        r=het_arch(resid_std,nlags=lag)
        arch_rows.append({"Lag":lag,"LM Statistic":r[0],"p-value":r[1]})
    display(pd.DataFrame(arch_rows))

In [ ]:
res_arch2 = modelo_arch2.std_resid.dropna()
plot_acf(res_arch2, lags=24)
plt.title("ACF - Standardized Residuals, ARCH(2)")
plt.show()
plot_pacf(res_arch2, lags=24, method="ywm")
plt.title("PACF - Standardized Residuals, ARCH(2)")
plt.show()
plot_acf(res_arch2**2, lags=24)
plt.title("ACF - Squared Standardized Residuals, ARCH(2)")
plt.show()

The ARCH(2) specification captures the main conditional-variance dynamics, but autocorrelation remains in the standardized residuals, motivating a re-specification of the conditional mean.

## 11. Conditional-Mean Re-specification

In [ ]:
modelo_ar1_arch2 = arch_model(y, x=X, mean="ARX", lags=1, vol="ARCH", p=2, dist="normal").fit(disp="off")
modelo_ar2_arch2 = arch_model(y, x=X, mean="ARX", lags=2, vol="ARCH", p=2, dist="normal").fit(disp="off")
comparacion_media = pd.DataFrame({
    "Model":["AR(0)-X-ARCH(2)","AR(1)-X-ARCH(2)","AR(2)-X-ARCH(2)"],
    "AIC":[modelo_arch2.aic,modelo_ar1_arch2.aic,modelo_ar2_arch2.aic],
    "BIC":[modelo_arch2.bic,modelo_ar1_arch2.bic,modelo_ar2_arch2.bic],
    "LogLik":[modelo_arch2.loglikelihood,modelo_ar1_arch2.loglikelihood,modelo_ar2_arch2.loglikelihood]
}).sort_values("AIC")
display(comparacion_media)

## 12. Final AR(2)-X-ARCH(2) Model

In [ ]:
print(modelo_ar2_arch2.summary())

## 13. Final Model Diagnostics

In [ ]:
r = modelo_ar2_arch2.std_resid.dropna()
print("=== Ljung-Box: standardized residuals ===")
display(acorr_ljungbox(r,lags=[6,12,18],return_df=True))
print("=== Ljung-Box: squared standardized residuals ===")
display(acorr_ljungbox(r**2,lags=[6,12,18],return_df=True))
rows=[]
for lag in [3,6,12]:
    t=het_arch(r,nlags=lag)
    rows.append({"Lag":lag,"LM Statistic":t[0],"p-value":t[1]})
print("=== ARCH-LM ===")
display(pd.DataFrame(rows))

In [ ]:
plot_acf(r,lags=24)
plt.title("ACF - Final Standardized Residuals")
plt.show()
plot_pacf(r,lags=24,method="ywm")
plt.title("PACF - Final Standardized Residuals")
plt.show()
plot_acf(r**2,lags=24)
plt.title("ACF - Final Squared Standardized Residuals")
plt.show()

## 14. Granger Causality Analysis

In [ ]:
df_granger = datos[["inflacion_mensual", "variacion_dolar"]].copy()
df_granger["delta_inflacion"] = df_granger["inflacion_mensual"].diff()
df_granger = df_granger[["delta_inflacion", "variacion_dolar"]].dropna()

g1 = grangercausalitytests(df_granger[["delta_inflacion","variacion_dolar"]], maxlag=6, verbose=False)
rows=[]
for lag in range(1,7):
    t=g1[lag][0]["ssr_ftest"]
    rows.append({"Lag":lag,"F Statistic":t[0],"p-value":t[1]})
print("Exchange-rate variation -> change in inflation")
display(pd.DataFrame(rows))

g2 = grangercausalitytests(df_granger[["variacion_dolar","delta_inflacion"]], maxlag=6, verbose=False)
rows=[]
for lag in range(1,7):
    t=g2[lag][0]["ssr_ftest"]
    rows.append({"Lag":lag,"F Statistic":t[0],"p-value":t[1]})
print("Change in inflation -> exchange-rate variation")
display(pd.DataFrame(rows))

Granger causality is interpreted as predictive precedence, not structural economic causality.

## 15. Parameter Stability: CUSUM

In [ ]:
df_cusum = datos[["Periodo","inflacion_mensual","variacion_dolar"]].copy()
df_cusum["delta_inflacion"] = df_cusum["inflacion_mensual"].diff()
df_cusum["delta_inf_lag1"] = df_cusum["delta_inflacion"].shift(1)
df_cusum["delta_inf_lag2"] = df_cusum["delta_inflacion"].shift(2)
df_cusum = df_cusum.dropna()
X_cusum = sm.add_constant(df_cusum[["delta_inf_lag1","delta_inf_lag2","variacion_dolar"]])
y_cusum = df_cusum["delta_inflacion"]
modelo_ols_cusum = sm.OLS(y_cusum,X_cusum).fit()
print(modelo_ols_cusum.summary())

In [ ]:
cusum_test = breaks_cusumolsresid(modelo_ols_cusum.resid, ddof=int(modelo_ols_cusum.df_model)+1)
print("CUSUM statistic:", cusum_test[0])
print("p-value:", cusum_test[1])
print("Critical values:", cusum_test[2])

In [ ]:
rr = recursive_olsresiduals(modelo_ols_cusum, skip=5, alpha=0.95)
cusum_path = rr[5]
cusum_ci = rr[6]
fechas_cusum = df_cusum["Periodo"].iloc[-len(cusum_path):].reset_index(drop=True)
fechas_band = fechas_cusum.iloc[1:].reset_index(drop=True)
plt.figure(figsize=(12,6))
plt.plot(fechas_cusum,cusum_path,label="CUSUM")
plt.plot(fechas_band,cusum_ci[0],linestyle="--",label="Lower bound")
plt.plot(fechas_band,cusum_ci[1],linestyle="--",label="Upper bound")
plt.axhline(0,linewidth=1)
plt.title("CUSUM - Conditional Mean Stability")
plt.xlabel("Period")
plt.ylabel("CUSUM")
plt.legend()
plt.tight_layout()
plt.show()

## 16. Summary

The final model is an **AR(2)-X-ARCH(2)** specification for changes in monthly inflation. Potential extensions include lagged-only exchange-rate specifications, VAR-based joint dynamics, and explicit structural-break or regime-switching models.